In [1]:
!pip -q install pypdf sentence-transformers faiss-cpu gradio numpy

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.6 MB/s eta 0:00:00
Installation complete.


In [2]:
import os
import re
import numpy as np
import faiss
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("Libraries imported.")

Libraries imported.


In [3]:
from google.colab import files

uploaded = files.upload()
PDF_DIR = Path("/content/university_pdfs")
PDF_DIR.mkdir(exist_ok=True)

for name, data in uploaded.items():
    path = PDF_DIR / name
    path.write_bytes(data)

print(f"Loaded {len(uploaded)} PDF file(s).")

Saving Generative_AI_RAG_Study_Material (1).pdf to Generative_AI_RAG_Study_Material (1).pdf
Loaded 1 PDF file(s).


In [4]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print("PDFs found:")
for p in pdf_files:
    print("-", p.name)

PDFs found:
- Generative_AI_RAG_Study_Material (1).pdf


In [5]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def load_pdfs(pdf_dir):
    documents = []
    for pdf_path in sorted(Path(pdf_dir).glob("*.pdf")):
        reader = PdfReader(str(pdf_path))
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            text = clean_text(text)
            if text:
                documents.append({
                    "text": text,
                    "source": pdf_path.name,
                    "page": page_num
                })
    return documents

pages = load_pdfs(PDF_DIR)
print(f"Extracted {len(pages)} non-empty pages.")
if pages:
    print(pages[0]["source"], "page", pages[0]["page"])

Extracted 2 non-empty pages.
Generative_AI_RAG_Study_Material (1).pdf page 1


In [6]:
def chunk_text(text, chunk_size=700, overlap=120):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

def make_chunks(pages, chunk_size=700, overlap=120):
    records = []
    for page in pages:
        for chunk in chunk_text(page["text"], chunk_size, overlap):
            records.append({
                "text": chunk,
                "source": page["source"],
                "page": page["page"]
            })
    return records

chunks = make_chunks(pages)
print(f"Created {len(chunks)} chunks.")
if chunks:
    print(chunks[0])

Created 2 chunks.
{'text': 'Generative AI and Retrieval-Augmented Generation (RAG) 1. Introduction to Generative AI Generative Artificial Intelligence (Generative AI) is a type of AI that can create new content such as text, images, audio, video, and code. It learns patterns from large datasets and uses those patterns to generate useful outputs. 2. Large Language Models (LLMs) Large Language Models are AI models trained on large collections of text. They can understand prompts and generate natural-language responses. Examples of capabilities include question answering, summarization, translation, and text generation. 3. Transformers Transformers are a neural-network architecture widely used in modern language models. They use attention mechanisms to identify relationships between words and other parts of a sequence. This helps models understand context efficiently. 4. Prompt Engineering Prompt engineering means writing clear instructions for an AI model. A good prompt can specify the t

In [7]:
# Multilingual model is useful if questions/documents contain Indian languages.
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(MODEL_NAME)

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index size:", index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index size: 2


In [8]:
def retrieve(query, k=5):
    if not chunks:
        return []
    q = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q, min(k, len(chunks)))

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            item = dict(chunks[idx])
            item["score"] = float(score)
            results.append(item)
    return results

In [9]:
def answer_question(query, k=5, threshold=0.30):
    results = retrieve(query, k)

    if not results or results[0]["score"] < threshold:
        return {
            "answer": "I could not find this information in the uploaded university documents.",
            "sources": []
        }

    # Use the strongest retrieved passages as grounded context.
    selected = [r for r in results if r["score"] >= threshold]

    # Simple extractive response: show the most relevant passages.
    parts = []
    for r in selected[:3]:
        parts.append(
            f"According to {r['source']} (page {r['page']}):\n{r['text']}"
        )

    return {
        "answer": "\n\n".join(parts),
        "sources": [(r["source"], r["page"], round(r["score"], 3)) for r in selected[:3]]
    }

In [10]:
import gradio as gr

def chat(query):
    if not query.strip():
        return "Please enter a question.", ""

    result = answer_question(query)
    sources = result["sources"]

    if not sources:
        return result["answer"], "No supporting source was found."

    source_text = "\n".join(
        f"- {doc}, page {page}, similarity score {score}"
        for doc, page, score in sources
    )
    return result["answer"], source_text

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        label="Ask about university regulations",
        placeholder="Example: What is the minimum attendance required?"
    ),
    outputs=[
        gr.Textbox(label="Grounded Answer"),
        gr.Textbox(label="Source Document / Page")
    ],
    title="University Regulation RAG Assistant",
    description="Answers are based only on the uploaded university PDF documents."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d26acb2fc92592a07.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
